# Can we rematch replies by meaning?

1. Count distinct archived opening replies per OP that were outside the published pairs, by reconstructed OP delta outcome.
2. Apply the published filters and count what remains.
3. For one published winning reply, rank eligible same-OP comparisons by semantic similarity and check the original match's rank.

Run with a Python kernel that can import the project's `cmv_data` package and has `pandas` and `sentence-transformers` installed. The database supplies metadata and the local v1 Parquet release supplies text. The default is a seeded training pilot. Set `SAMPLE_PER_GROUP = None` to use the full training pool. Results refer to archived replies and recorded awards.

In [15]:
from collections import defaultdict
from contextlib import closing
from pathlib import Path
from random import Random
import re

# builds the EDA tables. dict_row makes SQL rows accessible by column name.
import pandas as pd
from IPython.display import display
from psycopg.rows import dict_row

# db reads metadata from PostgreSQL. TextStore reads bodies from local Parquet.
from cmv_data import Database, TextStore

# Find the checkout from either the project root or a notebook subdirectory.
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "team-setup").is_dir() and (path / "pyproject.toml").is_file()
)
TEXT_DIR = REPO_ROOT / "project/data/processed/cmv-text/v1"

# Sampling sorted IDs with a fixed seed makes the pilot reproducible.
SEED = 42

SAMPLE_PER_GROUP = 3  # Additional training posts per group. None scans the full pool.

# always include this post. The source line identifies the paper's pair within it.
EXAMPLE_POST = "t3_18dj9h"
EXAMPLE_SOURCE_LINE = 2860

# Thresholds from the paper's paired analysis. The notebook approximates them
# using the archived discussion and our own word counts.
MIN_OPENING_WORDS = 50
MIN_CHALLENGERS = 10
MIN_EARLY_UNSUCCESSFUL_UNITS = 3

## Load the pilot data

Sample sorted training post IDs in three groups: posts with published pairs, posts with an OP delta label and no published pair, and remaining posts with no published pair. The walkthrough post is included explicitly. The groups help us inspect different cases. Their combined totals describe this pilot sample.

In [16]:
# Get the training post IDs and the two fields used to group them before sampling.
# The LEFT JOIN keeps posts whose OP-level delta label is missing.
POOL_SQL = '''
SELECT p.post_id, EXISTS (
    SELECT FROM cmv.pairs AS pair WHERE pair.post_id = p.post_id
) AS has_published_pairs, labels.delta_label
FROM cmv.posts AS p
LEFT JOIN cmv.op_labels AS labels USING (post_id)
WHERE p.original_split = 'train' AND p.content_available
ORDER BY p.post_id
'''

# Fetch parent links, timestamps, and authors for the chosen posts.
# ANY(%s) receives the list of sampled post IDs from Python.
COMMENT_SQL = '''
SELECT c.post_id, c.comment_id, c.original_parent_id, c.parent_comment_id,
       c.author, c.created_utc, c.content_available, p.author AS op_author
FROM cmv.comments AS c
JOIN cmv.posts AS p USING (post_id)
WHERE c.post_id = ANY(%s)
ORDER BY c.post_id, c.created_utc NULLS LAST, c.comment_id
'''

# Each published pair has a positive and negative branch. Position 0 identifies
# its first stored comment. The LEFT JOIN keeps the branch if that comment is missing.
PAIR_SQL = '''
SELECT t.post_id, t.pair_id, p.source_line, t.outcome, tc.comment_id
FROM cmv.threads AS t
JOIN cmv.pairs AS p USING (pair_id, post_id)
LEFT JOIN cmv.thread_comments AS tc
  ON tc.thread_id = t.thread_id AND tc.position = 0
WHERE t.post_id = ANY(%s)
ORDER BY t.post_id, t.pair_id, t.outcome
'''

# run these queries in a read-only transaction and close the connection afterward
with closing(Database(env_file=REPO_ROOT / ".env")) as db:
    with db.connection.transaction(), db.connection.cursor(row_factory=dict_row) as cursor:
        cursor.execute("SET TRANSACTION READ ONLY")
        cursor.execute(POOL_SQL)
        post_pool = cursor.fetchall()
        
        # Start with the fixed example, then sample each group without replacement
        rng = Random(SEED)
        post_ids = [EXAMPLE_POST]
        groups = defaultdict(list)
        pool_by_id = {row["post_id"]: row for row in post_pool}
        
        for row in post_pool:
            group = (
                "published_pairs" if row["has_published_pairs"]
                else "op_delta_without_pairs" if row["delta_label"] is True
                else "other_without_pairs"
            )
            
            # the example is already in post_ids, so exclude it from the random draw.
            if row["post_id"] != EXAMPLE_POST:
                groups[group].append(row["post_id"])
                
        for group in ("published_pairs", "op_delta_without_pairs", "other_without_pairs"):
            ids = groups[group]
            post_ids.extend(
                ids if SAMPLE_PER_GROUP is None
                else rng.sample(ids, min(SAMPLE_PER_GROUP, len(ids)))
            )
            
        # fetch metadata in batches and group it by post for the per-discussion analysis.
        cursor.execute(COMMENT_SQL, (post_ids,))
        comments_by_post = defaultdict(list)
        
        for row in cursor.fetchall():
            comments_by_post[row["post_id"]].append(row)
            
        cursor.execute(PAIR_SQL, (post_ids,))
        
        pairs_by_post = defaultdict(list)
        
        for row in cursor.fetchall():
            pairs_by_post[row["post_id"]].append(row)

# open the Parquet text release after the database transaction has closed.
store = TextStore(TEXT_DIR, expected_version="v1")
print("OP posts in the pilot:", len(post_ids))

OP posts in the pilot: 10


## Reconstruct recorded outcomes and apply the paper's filters

Follow a DeltaBot confirmation to its parent (the award comment) and then to the award recipient. Count awards by the OP to the opening challenger along that branch. A published label can fill a missing reconstruction for a published opening. Unpaired openings with no reconstructed award are labeled **no recorded delta**. An unresolved possible OP confirmation, or a positive OP label with no located award and no published winner, makes otherwise negative unpaired labels unknown for that post.

The paper's §4.1 uses at least 50 words in the opening, at least 10 challengers in the discussion, and at least three unsuccessful rooted path-units before the OP's last comment. Here a unit is the sequence of comments by the opening challenger along an archived root-to-leaf path. Identical sequences count once. We omit DeltaBot and identified award comments when building paths. The timing check uses the last challenger comment in each unit. Word counts use cleaned whitespace-separated text and challenger counts use distinct archived non-OP, non-bot authors. These are explicit reconstruction choices. We also require comparison openings to precede the OP's last comment.

[Original paper, §4.1](https://www.cs.cornell.edu/~cristian/pdfs/winning_arguments.pdf)

In [17]:
# Set aside bot and missing-author labels. The confirmation pattern distinguishes
# successful DeltaBot confirmations from other bot messages.
BOT = "deltabot"
NON_CHALLENGERS = {"", "[deleted]", "[removed]", BOT, "automoderator"}
CONFIRMED = re.compile(r"^confirmed\s*[-–—]\s*\d+\s+deltas?\s+awarded\s+to\b", re.I)
EDIT_LINE = re.compile(r"^\W*(edit|update)\b", re.I)


# normalize usernames for comparisons, including archived rows without an author.
def author(row):
    return (row.get("author") or "").casefold()


# Remove unavailable bodies, quoted lines, and lines that start with an edit marker.
# Join the remaining lines into a single string for counting words and embedding.
def clean(text):
    if not text or text.strip().casefold() in {"[deleted]", "[removed]"}:
        return ""
    return " ".join(
        line.strip() for line in text.splitlines()
        if not line.lstrip().startswith((">", "&gt;")) and not EDIT_LINE.match(line)
    ).strip()


def inspect_discussion(post_id, comments, pairs, bodies, op_delta_label):
    # index comments for parent lookups and find the timestamp of the OP's last comment
    by_id = {row["comment_id"]: row for row in comments}
    op = (comments[0]["op_author"] or "").casefold() if comments else ""
    last_op_reply = max(
        (row["created_utc"] for row in comments
         if op and author(row) == op and row["created_utc"] is not None),
        default=None,
    )
    
    # Estimate the number of challengers from distinct observed non-OP authors
    challengers = {author(row) for row in comments} - (NON_CHALLENGERS | {op})
    
    # One opening can appear in multiple pairs, so retain all its published outcomes.
    published = defaultdict(set)
    for row in pairs:
        if row["comment_id"]:
            published[row["comment_id"]].add(row["outcome"])

    # Build the reply tree. A root replies directly to the post. Other comments
    # are listed under their parent comment so we can follow the discussion downward.
    children = defaultdict(list)
    roots = []
    award_comments = set()
    recipients = set()
    unresolved = 0
    for row in comments:
        cid = row["comment_id"]
        if row["original_parent_id"] == post_id:
            if author(row) not in NON_CHALLENGERS | {op}:
                roots.append(cid)
        elif row["parent_comment_id"]:
            children[row["parent_comment_id"]].append(cid)
            
        # Award chain: DeltaBot reply -> award comment by the OP -> recipient comment.
        if author(row) != BOT:
            continue
        
        award = by_id.get(row["parent_comment_id"])
        
        # skip awards made by a known non-OP author
        if op and award and author(award) and author(award) != op:
            continue
        
        body = (bodies.get(cid) or "").strip().lstrip("*_").strip()
        if body and body.casefold() not in {"[deleted]", "[removed]"} and not CONFIRMED.match(body):
            continue
        
        # a possible confirmation without a complete chain leaves the award unresolved
        if not (op and award and author(award) == op and CONFIRMED.match(body)
                and award["parent_comment_id"] in by_id):
            unresolved += 1
            continue
        award_comments.add(award["comment_id"])
        recipients.add(award["parent_comment_id"])

    # a positive post-level label can indicate an award without identifying its branch.
    # If neither the archive nor the published pairs locate a winner, keep that uncertainty.
    op_delta_without_located_award = (
        op_delta_label is True
        and not recipients
        and not any("positive" in outcomes for outcomes in published.values())
    )
    
    uncertain_awards = bool(unresolved) or op_delta_without_located_award
    opening_rows = []
    unsuccessful_early_units = 0
    published_disagreements = []
    
    for root in roots:
        root_row = by_id[root]
        root_author = author(root_row)
        path_units = set()
        
        # each stack entry stores our current node, the root author's comments on this
        # path, and the visited IDs. The visited set guards against cycles.
        
        pending = [(root, (root,), frozenset({root}))]
        while pending:
            
            cid, unit, seen = pending.pop()
            next_replies = [
                child for child in children[cid]
                if child not in seen and author(by_id[child]) != BOT
            ]
            
            # at a leaf, save the root author's sequence. A set deduplicates identical
            # sequences reached through different branches of the reply tree.
            if not next_replies:
                path_units.add(unit)
                
            for child in next_replies:
                # Follow every eligible child but only add the root author's own turns to the unit.
                challenger_turn = author(by_id[child]) == root_author and child not in award_comments
                pending.append((child, (*unit, child) if challenger_turn else unit, seen | {child}))

        # The opening counts as a recorded win if an OP award recipient is in any of its
        # root-author path units. Keep track of disagreements with the published labels.
        outcomes = published[root]
        recorded = any(recipients.intersection(unit) for unit in path_units)
        if (recorded and outcomes == {"negative"}) or (
            not recorded and "positive" in outcomes
        ):
            published_disagreements.append(root)
            
        # A published positive can supply a win even if the bot chain is missing.
        # Unknown means we cannot rule out an award based on the available evidence.
        if recorded or "positive" in outcomes:
            delta_status = "delta"
        elif (uncertain_awards and outcomes != {"negative"}) or not op:
            delta_status = "unknown"
        else:
            delta_status = "no_recorded_delta"

        opening = clean(bodies.get(root)) if root_row["content_available"] else ""
        
        # Count whitespace-separated words in the cleaned opening and check its timestamp.
        word_count = len(opening.split())
        root_time = root_row["created_utc"]
        before_last_op = (
            root_time is not None and last_op_reply is not None and root_time < last_op_reply
        )
        
        if word_count >= MIN_OPENING_WORDS:
            for unit in path_units:
                # Use the final root-author turn to decide whether the unit ended before the OP's
                # last reply. Count only units we can classify as unsuccessful.
                end_time = by_id[unit[-1]]["created_utc"]
                # When a published win has no located award, its winning unit is uncertain.
                if (not recipients.intersection(unit)
                        and (delta_status == "no_recorded_delta" or (recorded and not uncertain_awards))
                        and end_time is not None and last_op_reply is not None
                        and end_time < last_op_reply):
                    unsuccessful_early_units += 1
                    
        opening_rows.append({
            "post_id": post_id,
            "comment_id": root,
            "text": opening,
            "word_count": word_count,
            "before_last_op_reply": before_last_op,
            "delta_status": delta_status,
            "published": bool(outcomes),
        })

    # Save post-level diagnostics so we can see why a discussion passed or failed.
    diagnostics = {
        "post_id": post_id,
        "published_pairs": len({row["pair_id"] for row in pairs}),
        "observed_challengers": len(challengers),
        "early_unsuccessful_path_units": unsuccessful_early_units,
        "unresolved_bot_confirmations": unresolved,
        "op_delta_without_located_award": op_delta_without_located_award,
        "published_disagreements": published_disagreements,
    }
    diagnostics["passes_discussion_filters"] = (
        len(challengers) >= MIN_CHALLENGERS
        and unsuccessful_early_units >= MIN_EARLY_UNSUCCESSFUL_UNITS
    )
    # Apply the discussion thresholds and the opening-length cutoff. Comparison
    # openings also need to precede the OP's last reply.
    for row in opening_rows:
        row["passes_filters"] = (
            diagnostics["passes_discussion_filters"]
            and row["word_count"] >= MIN_OPENING_WORDS
            and (row["delta_status"] == "delta" or row["before_last_op_reply"])
        )
    return opening_rows, diagnostics

## Count remaining opening replies per OP

The first counts use all unmatched openings with usable text. The second counts apply the word, timing, and discussion-level checks. Counts are by distinct opening, so a reply that appeared in multiple original pairs is excluded once.

In [18]:
# Collect one row per opening and one diagnostic row per OP discussion.
candidate_rows = []
diagnostic_rows = []

for post_id in post_ids:
    text_rows = store.get_comments(post_id)
    # Join the Parquet bodies to PostgreSQL metadata by comment ID within this post.
    bodies = {row["comment_id"]: row["body"] for row in text_rows}
    openings, diagnostics = inspect_discussion(
        post_id, comments_by_post[post_id], pairs_by_post[post_id], bodies,
        pool_by_id[post_id]['delta_label'],
    )
    candidate_rows.extend(openings)
    diagnostics["op_delta_label"] = pool_by_id[post_id]["delta_label"]
    diagnostic_rows.append(diagnostics)

replies = pd.DataFrame(candidate_rows)
diagnostics = pd.DataFrame(diagnostic_rows).set_index("post_id")

# For the leftover counts, use openings outside the published pairs with usable text.
leftovers = replies.loc[~replies["published"] & replies["text"].ne("")]


def per_post_counts(frame, prefix):
    # Count openings by post and outcome, then turn the outcome groups into columns.
    # Reindex so posts and outcome categories with zero matches still appear.
    counts = frame.groupby(["post_id", "delta_status"]).size().unstack(fill_value=0)
    return counts.reindex(
        index=post_ids, columns=["delta", "no_recorded_delta", "unknown"], fill_value=0
    ).add_prefix(prefix)


# Attach the counts for all leftovers and for the subset that passes the filters.
eda_counts = (
    diagnostics.join(per_post_counts(leftovers, "before_"))
    .join(per_post_counts(leftovers.loc[leftovers["passes_filters"]], "after_"))
)
# A fully new pair needs both a leftover winner and a leftover comparison
# under the same OP. Unknown outcomes do not count toward either side.
for stage in ("before", "after"):
    eda_counts[f"{stage}_can_form_new_pair"] = (
        eda_counts[f"{stage}_delta"].gt(0) & eda_counts[f"{stage}_no_recorded_delta"].gt(0)
    )
display(eda_counts)
print("OP posts with both unmatched outcomes before filters:", eda_counts["before_can_form_new_pair"].sum())
print("OP posts with both unmatched outcomes after filters:", eda_counts["after_can_form_new_pair"].sum())

,published_pairs,observed_challengers,early_unsuccessful_path_units,unresolved_bot_confirmations,op_delta_without_located_award,published_disagreements,passes_discussion_filters,op_delta_label,before_delta,before_no_recorded_delta,before_unknown,after_delta,after_no_recorded_delta,after_unknown,before_can_form_new_pair,after_can_form_new_pair
post_id,,,,,,,,,,,,,,,,
t3_18dj9h,2,92,24,0,False,[],True,True,0,37,0,0,19,0,False,False
t3_1l6so4,1,120,35,0,False,[t1_cbwanvb],True,True,0,48,0,0,28,0,False,False
t3_1daa73,1,11,6,0,False,[],True,True,0,6,0,0,3,0,False,False
t3_22apfm,1,14,6,0,False,[t1_cglkcag],True,True,0,8,0,0,4,0,False,False
t3_1q1ykh,0,16,0,0,True,[],False,True,0,0,5,0,0,0,False,False
t3_1nusf8,0,9,0,0,True,[],False,True,0,0,6,0,0,0,False,False
t3_1idapv,0,11,0,0,True,[],False,True,0,0,7,0,0,0,False,False
t3_2htyyi,0,63,14,0,False,[],True,False,0,34,0,0,11,0,False,False
t3_1f2g72,0,7,2,0,False,[],False,None,1,5,0,0,0,0,True,False


OP posts with both unmatched outcomes before filters: 1
OP posts with both unmatched outcomes after filters: 0


## Semantic top five for the published pair ending in #2860

Rank the eligible openings with no recorded delta in the same OP discussion. Include eligible published comparisons so the original match can appear in the ranking. Use cosine similarity of normalized MiniLM embeddings. Token counts use the encoder's tokenizer before truncation and include special tokens. The `truncated` column identifies openings that exceed the model's input limit.

[MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

In [19]:
from sentence_transformers import SentenceTransformer

# Pin the exact model revision so both ranking methods use the same weights.
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

# Look up the published pair's positive and negative opening IDs by source line.
example_pair = {
    row["outcome"]: row["comment_id"]
    for row in pairs_by_post[EXAMPLE_POST]
    if row["source_line"] == EXAMPLE_SOURCE_LINE
}

example_replies = replies.loc[replies["post_id"].eq(EXAMPLE_POST)].set_index("comment_id")
winner = example_replies.loc[example_pair["positive"]]
published_match_id = example_pair["negative"]

# Keep eligible same-OP openings with no recorded delta. Include the published
# comparison when it qualifies so we can see where the model ranks it.
eligible = example_replies.loc[
    example_replies["passes_filters"] & example_replies["delta_status"].eq("no_recorded_delta")
].copy()

if not winner["passes_filters"] or eligible.empty:
    raise ValueError("The chosen example has no eligible semantic comparison under these filters.")

model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device="cpu")
texts = [winner["text"], *eligible["text"]]
token_limit = model.max_seq_length

# Measure the original token lengths with special tokens and with truncation off.
# Using the model's tokenizer matters because one word can use several tokens.
full_token_ids = model.tokenizer(
    texts, add_special_tokens=True, truncation=False, verbose=False
)["input_ids"]
winner_token_count, *candidate_token_counts = map(len, full_token_ids)
eligible["token_count"] = candidate_token_counts
eligible["truncated"] = eligible["token_count"].gt(token_limit)

# The encoder applies its normal input cap here. Row 0 is the winner.
# With unit-length embeddings, each candidate's dot product with row 0 is cosine similarity.
embeddings = model.encode(texts, normalize_embeddings=True)
eligible["cosine_similarity"] = embeddings[1:] @ embeddings[0]

# Sort best scores first and use comment ID to break a tie reproducibly.
ranked = eligible.reset_index().sort_values(
    ["cosine_similarity", "comment_id"], ascending=[False, True]
).reset_index(drop=True)

ranked.index = ranked.index + 1
ranked.index.name = "rank"

# This flag singles out the specific comparison from the published example pair.
ranked["original_match"] = ranked["comment_id"].eq(published_match_id)
top_five = ranked.head(5)

print("Model input limit:", token_limit, "tokens")
display(pd.DataFrame([{
    "winning_comment_id": winner.name,
    "word_count": winner["word_count"],
    "token_count": winner_token_count,
    "truncated": winner_token_count > token_limit,
}]))
display(top_five[[
    "comment_id", "cosine_similarity", "original_match", "published", "word_count",
    "token_count", "truncated",
]])
# Look up the published comparison's rank across the entire eligible pool.
match_ranks = ranked.index[ranked["original_match"]].tolist()
print("Published openings in the top five:", int(top_five["published"].sum()))
print("Published comparison:", published_match_id)

if match_ranks:
    print("Semantic rank:", match_ranks[0], "| In the top five:", match_ranks[0] <= 5)
else:
    print("The published comparison was outside this pilot's eligible pool.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15428.50it/s]


Model input limit: 256 tokens


,winning_comment_id,word_count,token_count,truncated
0,t1_c8dtv3e,572,695,True


,comment_id,cosine_similarity,original_match,published,word_count,token_count,truncated
rank,,,,,,,
1,t1_c8r6azm,0.696822,False,False,113,162,False
2,t1_c8l7u7y,0.686363,True,True,527,649,True
3,t1_c8e01qw,0.660499,False,False,282,411,True
4,t1_c8dtkpu,0.634333,False,False,172,212,False
5,t1_c8dtq8n,0.624842,False,False,73,106,False


Published openings in the top five: 1
Published comparison: t1_c8l7u7y
Semantic rank: 2 | In the top five: True


In [20]:
# Use the same example ID to fetch the OP's title and full body from Parquet.
example_op = store.get_post(EXAMPLE_POST)
print(example_op["title"])
print()
print(example_op["body"])

I believe there are no valid arguments for eating meat, other than "I like the taste" - CMV

I'm a vegetarian, and have been all my life, so I'm well aware i might not have chosen to be if my parents hadn't brought me up that way. I'm fairly sure of my beliefs that eating meat is wrong for various reason including animal welfare, environmental impact and inefficiency/inhumanity of the process, but I'm also not a vegan, which seems to be the most logical/valid position to hold.
I also believe that a vegetarian diet is no less healthy than an omnivorous diet, and in fact can be more healthy in many ways.

CMV!

EDIT: This is a good discussion! Please don't downvote people who support the vegetarian viewpoint just because you disagree. Likewise, veggies, don't downvote people because they support eating meat! Downvotes are for things that aren't relevant to the topic.


## Compare rankings with full-text chunks

Use the same model revision and eligible replies. Group consecutive sentences into chunks of up to 112 input tokens, including special tokens. This keeps the passages short enough to inspect. Split unusually long sentences at token boundaries, preferring a word break. All cleaned text goes into the chunks.

- **Strongest match** is the highest cosine similarity between a winning-reply chunk and a candidate chunk. A single shared argument can rank highly.
- **Winner coverage** is the average best candidate similarity across all winning-reply chunks. It measures how broadly the candidate matches the winner's different points.

The tables show both ranks alongside the original single-input rank. The passage table shows the strongest chunk pair per candidate so we can inspect what the scores picked up.

In [21]:
# Aim for passages of at most 112 tokens while respecting the model's input limit.
CHUNK_TOKEN_LIMIT = min(112, token_limit)

# Use sentence-ending punctuation followed by whitespace as a simple split rule.
SENTENCE_BREAK = re.compile(r"(?<=[.!?])\s+")
NUTRITION_EXAMPLE = "t1_c8dtkpu"


def token_count(text):
    # Count the input tokens a passage would use, including the model's special tokens.
    return len(model.tokenizer(
        text, add_special_tokens=True, truncation=False, verbose=False
    )["input_ids"])


def chunk_reply(text):
    """Pack adjacent sentences and split any sentence that exceeds the chunk limit."""
    chunks = []
    current = ""
    # Reserve room for special tokens before calculating how much text can fit.
    content_budget = CHUNK_TOKEN_LIMIT - model.tokenizer.num_special_tokens_to_add(pair=False)
    for sentence in SENTENCE_BREAK.split(text.strip()):
        if not sentence:
            continue
        # Try adding the next sentence to the current chunk. If it fits, keep packing.
        combined = f"{current} {sentence}".strip()
        if token_count(combined) <= CHUNK_TOKEN_LIMIT:
            current = combined
            continue
        if current:
            chunks.append(current)
        # For an oversized sentence, use tokenizer offsets to find a character cut
        # and move back to a word boundary when possible. Then process the remainder.
        while token_count(sentence) > CHUNK_TOKEN_LIMIT:
            offsets = model.tokenizer(
                sentence, add_special_tokens=False, truncation=False,
                return_offsets_mapping=True, verbose=False
            )["offset_mapping"]
            cut = offsets[content_budget - 1][1]
            word_break = sentence.rfind(" ", 0, cut + 1)
            if cut < len(sentence) and not sentence[cut].isspace() and word_break > 0:
                cut = word_break
            chunks.append(sentence[:cut].strip())
            sentence = sentence[cut:].lstrip()
        current = sentence
    # Store the final chunk, which may still have room left.
    if current:
        chunks.append(current)
    return chunks


# Chunk the same winner and candidates used in the original ranking.
chunks_by_id = {
    comment_id: chunk_reply(text)
    for comment_id, text in [(winner.name, winner["text"]), *eligible["text"].items()]
}
# Embed identical passages only once, then map each passage back to its vector.
chunk_texts = list(dict.fromkeys(chunk for chunks in chunks_by_id.values() for chunk in chunks))
chunk_vectors = model.encode(
    chunk_texts, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False
)
chunk_index = {chunk: i for i, chunk in enumerate(chunk_texts)}
winning_chunks = chunks_by_id[winner.name]
winner_vectors = chunk_vectors[[chunk_index[chunk] for chunk in winning_chunks]]

# Keep the single-input ranks for the side-by-side comparison.
original_ranks = ranked.reset_index().set_index("comment_id")["rank"]

chunk_results = []
for comment_id in eligible.index:
    candidate_chunks = chunks_by_id[comment_id]
    candidate_vectors = chunk_vectors[[chunk_index[chunk] for chunk in candidate_chunks]]
    
    # Build the cosine similarity matrix for one candidate. Rows are winner chunks,
    # columns are candidate chunks, and each cell compares one pair of passages.
    similarities = winner_vectors @ candidate_vectors.T
    
    # Take each row's maximum. Each winner chunk can choose any candidate chunk,
    # and several rows can choose the same one.
    per_winner, best_candidate_indices = similarities.max(dim=1)
    
    # Find the row with the highest overall match and recover both passage indices.
    best_winner = int(per_winner.argmax())
    best_candidate = int(best_candidate_indices[best_winner])
    
    chunk_results.append({
        "comment_id": comment_id,
        "original_rank": int(original_ranks[comment_id]),
        
        # Strongest similarity takes the maximum of the row maxima. Coverage averages
        # those same maxima with equal weight for each winning chunk.
        "strongest_similarity": float(per_winner[best_winner]),
        "winner_coverage": float(per_winner.mean()),
        "candidate_chunks": len(candidate_chunks),
        "original_match": comment_id == published_match_id,
        "winning_passage": winning_chunks[best_winner],
        "candidate_passage": candidate_chunks[best_candidate],
    })

chunk_ranked = pd.DataFrame(chunk_results)

# Rank candidates separately by the strongest match and by how broadly they
# cover the winner. Break score ties by comment ID.
for metric in ("strongest_similarity", "winner_coverage"):
    ordered = chunk_ranked.sort_values([metric, "comment_id"], ascending=[False, True])
    chunk_ranked.loc[ordered.index, f"{metric}_rank"] = range(1, len(ordered) + 1)
    chunk_ranked[f"{metric}_rank"] = chunk_ranked[f"{metric}_rank"].astype(int)

rank_columns = [
    "comment_id", "original_match", "original_rank", "strongest_similarity_rank",
    "winner_coverage_rank", "strongest_similarity", "winner_coverage", "candidate_chunks",
]
chunk_top_five = chunk_ranked.nsmallest(5, "strongest_similarity_rank")
coverage_top_five = chunk_ranked.nsmallest(5, "winner_coverage_rank")
print("Winning reply chunks:", len(winning_chunks))
print("Maximum chunk length:", max(map(token_count, chunk_texts)), "tokens")
print("Top five by strongest shared passage")
display(chunk_top_five[rank_columns])
print("Top five by coverage of the winning reply")
display(coverage_top_five[rank_columns])

# Show the passages behind the highest scores and also include the published
# comparison and the nutrition reply so we can inspect them directly.
review_ids = set(chunk_top_five["comment_id"]) | {published_match_id, NUTRITION_EXAMPLE}
passage_review = chunk_ranked.loc[chunk_ranked["comment_id"].isin(review_ids)].sort_values(
    "strongest_similarity_rank"
)


Winning reply chunks: 8
Maximum chunk length: 112 tokens
Top five by strongest shared passage


,comment_id,original_match,original_rank,strongest_similarity_rank,winner_coverage_rank,strongest_similarity,winner_coverage,candidate_chunks
11,t1_c8e01qw,False,3,1,5,0.743975,0.526645,4
0,t1_c8dtg3d,False,10,2,3,0.725128,0.545217,3
13,t1_c8e1kul,False,9,3,2,0.715022,0.553947,6
16,t1_c8l7u7y,True,2,4,10,0.704244,0.501082,7
4,t1_c8dtky8,False,17,5,8,0.697773,0.502589,4


Top five by coverage of the winning reply


,comment_id,original_match,original_rank,strongest_similarity_rank,winner_coverage_rank,strongest_similarity,winner_coverage,candidate_chunks
12,t1_c8e03jy,False,8,8,1,0.646025,0.557337,4
13,t1_c8e1kul,False,9,3,2,0.715022,0.553947,6
0,t1_c8dtg3d,False,10,2,3,0.725128,0.545217,3
3,t1_c8dtkpu,False,4,6,4,0.670355,0.536886,3
11,t1_c8e01qw,False,3,1,5,0.743975,0.526645,4


In [22]:
# best compared chunk to the wining passage

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

print("Strongest matching passages, including the published and nutrition-related replies")
display(passage_review[[
    "comment_id", "strongest_similarity_rank", "original_match", "winning_passage", "candidate_passage"
]])


Strongest matching passages, including the published and nutrition-related replies


,comment_id,strongest_similarity_rank,original_match,winning_passage,candidate_passage
11,t1_c8e01qw,1,False,"Okay, lets start with your arguments, which are really more talking points. To eliminate the crux of your first item, animal welfare, I'd point out that while this is an issue, especially when we see clearly egregious mistreatment of animals, it has no impact on your question. The fact that people can mistreat animals in a farm setting should have no direct impact on whether you eat meat in general.","""my beliefs that eating meat is wrong for various reason including animal welfare, environmental impact and inefficiency/inhumanity"" I can see you stating a couple of your reasons. Here are my responses. First, What about plant welfare? I am stating that animals AND plants feel pain, so to care for the welfare of animals and plants we need to treat them the same. If you avoid meat, avoid carrots. Ect. Only eat fruits? Or, in my opinion, eat both plants and animals."
0,t1_c8dtg3d,2,False,"Certain people will experience a lack of energy more than others on this diet, certain people will experience anemia or calcium or b vitamin deficiencies more than others on a vegetarian diet - they should not feel uncomfortable or morally sick for wanting to live a life where they are living the way they were designed to live. So now lets ask why we should eat meat. I eat meat because it not only tastes good, but I feel more energetic when I'm eating meat. I get anemic without meat.","The most valid point in my view that eating meat is ethical, is that our bodies evolved to do it. We needed meat to get to where we are as a species. There are many valid reasons why we probably eat too much meat, or don't treat the animals we eat with enough compassion, but eating meat per se is something we've always done as a species. Biologically-speaking, we have evolved complex machinery in order to extract a great deal of energy, protein and iron from red meat."
13,t1_c8e1kul,3,False,"Okay, lets start with your arguments, which are really more talking points. To eliminate the crux of your first item, animal welfare, I'd point out that while this is an issue, especially when we see clearly egregious mistreatment of animals, it has no impact on your question. The fact that people can mistreat animals in a farm setting should have no direct impact on whether you eat meat in general.","I feel that, especially if you buy your edible goods (flora and fauna) from farmers as close to your home as possible, from people who care as much for their flora and fauna as possible (as most farmers do, despite their checkered reputation in the eyes of media outlets), then you're on the right track. I also encourage those of us who DO eat meat to recognize that animals are not all choice cuts. Not all of a cow is Sirloin steak, not all of a chicken is white meat."
16,t1_c8l7u7y,4,True,"Okay, lets start with your arguments, which are really more talking points. To eliminate the crux of your first item, animal welfare, I'd point out that while this is an issue, especially when we see clearly egregious mistreatment of animals, it has no impact on your question. The fact that people can mistreat animals in a farm setting should have no direct impact on whether you eat meat in general.","Ok, here's my shot! Firstly, I agree with your environmental arguments, but let's leave them aside for now. Secondly, animal welfare is important and we don't do enough on it. Personally, I think every slaughterhouse and farm should be mandated to upload a video of their conditions to the internet every year. That way people could see what was going on and judge for themselves if the animals were being treated well. Animal cruelty is in no way acceptable. However eating meat doesn't have to be cruel."
4,t1_c8dtky8,5,False,"Certain people will experience a lack of energy more than others on this diet, certain people will experience anemia or calcium or b vitamin deficiencies more than othe

In [23]:
# Fetch the published comparison's full archived body for a side-by-side reading.
published_candidate_body = store.attach_comment_text([
    {"post_id": EXAMPLE_POST, "comment_id": published_match_id}
])[0]["body"]
print(f"Published comparison ({published_match_id})")
print()
print(published_candidate_body)

Published comparison (t1_c8l7u7y)

Ok, here's my shot!

Firstly, I agree with your environmental arguments, but let's leave them aside for now.  Secondly, animal welfare is important and we don't do enough on it.  Personally, I think every slaughterhouse and farm should be mandated to upload a video of their conditions to the internet every year.  That way people could see what was going on and judge for themselves if the animals were being treated well.  Animal cruelty is in no way acceptable.  

However eating meat doesn't have to be cruel.  Our ancestors have hunted other animals for millions of years and probably wouldn't have survived without doing so.  The native Americans were said to have worshiped nature and treated it with respect.  They were supposed to have used every part of the buffalo, including all the bones, either to eat or use as tools or make into clothes.  

My point is that just because you kill something doesn't mean you don't have to respect it or treat it well 

## Compare politeness in the openings and the matched passages

Polite BERT scores each sentence using four labels: **not polite (0)**, **neutral (1)**, **somewhat polite (2)**, and **polite (3)**. Multiply each label's score by its model probability and sum them to get a sentence score. Average sentence scores equally to get an opening or passage score. For any sentence over the model's 128-token training length, score consecutive token windows and average them back into one sentence score.

A **positive difference** means the model scored the winning side as more polite. The passage comparison uses the strongest matched passages MiniLM already selected. The opening comparison uses the complete cleaned reply. These are exploratory model scores. The passages can share a topic and give different reasons, and the politeness scores still need a blind review on CMV text.

[Polite BERT model card](https://huggingface.co/NOVA-vision-language/polite_bert)

In [24]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

POLITENESS_MODEL = "NOVA-vision-language/polite_bert"
POLITENESS_REVISION = "9ab273fb2a166f2bfd6cfe9030a75d47e9343709"
POLITENESS_TOKEN_LIMIT = 128
POLITENESS_BATCH_SIZE = 16

politeness_tokenizer = AutoTokenizer.from_pretrained(
    POLITENESS_MODEL, revision=POLITENESS_REVISION
)
politeness_model = AutoModelForSequenceClassification.from_pretrained(
    POLITENESS_MODEL, revision=POLITENESS_REVISION, use_safetensors=True
).eval()

# Look up the weight of each output class by its checkpoint label.
label_weights = {"NOT_POLITE": 0, "NEUTRAL": 1, "SOMEWHAT_POLITE": 2, "POLITE": 3}
id_to_label = politeness_model.config.id2label
if set(id_to_label.values()) != set(label_weights):
    raise ValueError(f"Unexpected politeness labels: {id_to_label}")
weights = torch.tensor(
    [label_weights[id_to_label[i]] for i in range(len(id_to_label))], dtype=torch.float32
)


def sentence_list(text):
    """Use the notebook's punctuation-based sentence split."""
    return [sentence for sentence in SENTENCE_BREAK.split(text.strip()) if sentence]


def score_politeness(texts):
    """Batch unique sentences and return an equally weighted sentence mean per text."""
    sentences_by_text = {text: sentence_list(text) for text in dict.fromkeys(texts)}
    unique_sentences = list(dict.fromkeys(
        sentence for sentences in sentences_by_text.values() for sentence in sentences
    ))
    if not unique_sentences:
        return {text: None for text in sentences_by_text}

    sentence_scores = defaultdict(list)
    with torch.inference_mode():
        for start in range(0, len(unique_sentences), POLITENESS_BATCH_SIZE):
            sentences = unique_sentences[start:start + POLITENESS_BATCH_SIZE]

            # Ask the tokenizer for every window, including text past the first 128 tokens.
            # It also adds BERT's special tokens and pads the windows for batching.
            windows = politeness_tokenizer(
                sentences, max_length=POLITENESS_TOKEN_LIMIT, truncation=True,
                return_overflowing_tokens=True, stride=0, padding=True, return_tensors="pt"
            )
            # Map each window back to its input sentence. The model only needs the tensors.
            sentence_indices = windows.pop("overflow_to_sample_mapping").tolist()
            for offset in range(0, len(sentence_indices), POLITENESS_BATCH_SIZE):
                end = offset + POLITENESS_BATCH_SIZE
                batch = {name: tensor[offset:end] for name, tensor in windows.items()}
                probabilities = politeness_model(**batch).logits.softmax(dim=-1)
                scores = (probabilities @ weights).tolist()
                for sentence_index, score in zip(sentence_indices[offset:end], scores):
                    sentence_scores[sentences[sentence_index]].append(score)

    # Average split windows first so a long sentence counts as one sentence.
    sentence_means = {
        sentence: sum(scores) / len(scores) for sentence, scores in sentence_scores.items()
    }
    return {
        text: sum(sentence_means[sentence] for sentence in sentences) / len(sentences)
        if sentences else None
        for text, sentences in sentences_by_text.items()
    }

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19940.28it/s]


In [25]:
# Score complete cleaned openings and MiniLM's selected passages with the same model.
politeness_by_text = score_politeness([
    winner["text"], *eligible["text"],
    *passage_review["winning_passage"], *passage_review["candidate_passage"],
])
winner_opening_politeness = politeness_by_text[winner["text"]]
opening_politeness = eligible["text"].map(politeness_by_text)

# Keep the original-match flag and similarity rank visible when comparing scores.
politeness_review = passage_review[[
    "comment_id", "strongest_similarity_rank", "original_match",
]].copy()
politeness_review["winner_opening_politeness"] = winner_opening_politeness
politeness_review["candidate_opening_politeness"] = politeness_review["comment_id"].map(
    opening_politeness
)
politeness_review["opening_gap"] = (
    politeness_review["winner_opening_politeness"]
    - politeness_review["candidate_opening_politeness"]
)
politeness_review["winner_passage_politeness"] = passage_review["winning_passage"].map(
    politeness_by_text
)
politeness_review["candidate_passage_politeness"] = passage_review["candidate_passage"].map(
    politeness_by_text
)
politeness_review["passage_gap"] = (
    politeness_review["winner_passage_politeness"]
    - politeness_review["candidate_passage_politeness"]
)
display(politeness_review.round(3))

,comment_id,strongest_similarity_rank,original_match,winner_opening_politeness,candidate_opening_politeness,opening_gap,winner_passage_politeness,candidate_passage_politeness,passage_gap
11,t1_c8e01qw,1,False,0.966,1.000,-0.034,1.000,1.000,-0.000
0,t1_c8dtg3d,2,False,0.966,1.000,-0.034,0.788,1.000,-0.212
13,t1_c8e1kul,3,False,0.966,1.000,-0.033,1.000,0.997,0.003
16,t1_c8l7u7y,4,True,0.966,0.998,-0.032,1.000,1.000,-0.000
4,t1_c8dtky8,5,False,0.966,0.999,-0.033,0.788,1.000,-0.212
3,t1_c8dtkpu,6,False,0.966,1.000,-0.034,1.000,1.000,0.000
